In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")); from parallel_compute import *

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

Inputs

In [2]:
features_ranked_ordered = pd.read_parquet(variables.FEATURES_RANKED_ORDERED).iloc[:variables.FEATURES_UNIQUE_RANKED_LIMIT]
features = pd.read_parquet(variables.FEATURES_DATASET_FOR_SELECTION_PATH)

Subset sample

In [ ]:
# Randomly sample index
seed = np.random.default_rng(43)
index = seed.choice(len(features), size=variables.FEATURE_SELECTION_SUBSAMPLE_AMOUNT, replace=False)
index.sort()

features = features.iloc[index]

Core logic 

Iterates over n number of features

In [4]:
import warnings

def compute_correlations(item):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pearson = features.corrwith(features[item], method="pearson").abs().drop(item).fillna(0)
        spearman = features.corrwith(features[item], method="spearman").abs().drop(item).fillna(0)
    return pearson, spearman

corr_results = run_parallel(function=compute_correlations, data=features_ranked_ordered.index.tolist())


Initialising workers..:   0%|          | 0/5 [00:00<?, ?step/s]

os=linux cpu_count=48 available_memory_gb=176.42 memory_per_worker_gb=3.72 max_assigned_workers=45


Initialising workers..: 100%|██████████| 5/5 [00:49<00:00,  9.88s/step]


In [5]:
pd.DataFrame(corr_results)[:3]

,0,1
0,nsw_price_asinh_lag_2 0.92...,nsw_price_asinh_lag_2 0.94...
1,nsw_price_asinh_lag_1 0.92...,nsw_price_asinh_lag_1 0.94...
2,nsw_price_asinh_lag_1 0.88...,nsw_price_asinh_lag_1 0.91...


Clean up

In [6]:
LINEAR_CORR_THRESHOLD = 0.95
NONLINEAR_CORR_THRESHOLD = 0.95

unique_features = []
duplicate_features = []

for i, feature in enumerate(features_ranked_ordered.index):
    pearson_corr, spearman_corr = corr_results[i]

    is_linear_duplicate = (pearson_corr[unique_features] > LINEAR_CORR_THRESHOLD).any()
    is_nonlinear_duplicate = (spearman_corr[unique_features] > NONLINEAR_CORR_THRESHOLD).any()
    is_duplicate = bool(is_linear_duplicate or is_nonlinear_duplicate)


    if is_duplicate:
        duplicate_features.append(feature)
    else:
        unique_features.append(feature)
        
    del pearson_corr, spearman_corr

del corr_results


In [7]:
features_ranked_ordered_duplicate = features_ranked_ordered.loc[duplicate_features].reset_index().rename(columns={"index": "feature"})
features_ranked_ordered_unique = features_ranked_ordered.loc[unique_features].reset_index().rename(columns={"index": "feature"})
features_ranked_ordered_unique.index = features_ranked_ordered_unique['feature']
features_ranked_ordered_unique = features_ranked_ordered_unique.drop(columns='feature')

In [8]:
features.shape

(20000, 637)

In [9]:
features_unique_data = features[unique_features]

In [10]:
features_unique_data.shape

(20000, 466)

In [11]:
features_ranked_ordered_unique.shape

(466, 96)

View

In [12]:
display(features_ranked_ordered_unique[:10])
display(features_ranked_ordered_duplicate[:10])
print(len(features_ranked_ordered_duplicate))

,target_h1,target_h2,target_h3,target_h4,target_h5,target_h6,target_h7,target_h8,target_h9,target_h10,target_h11,target_h12,target_h13,target_h14,target_h15,target_h16,target_h17,target_h18,target_h19,target_h20,target_h21,target_h22,target_h23,target_h24,target_h25,target_h26,target_h27,target_h28,target_h29,target_h30,target_h31,target_h32,target_h33,target_h34,target_h35,target_h36,target_h37,target_h38,target_h39,target_h40,target_h41,target_h42,target_h43,target_h44,target_h45,target_h46,target_h47,target_h48,target_h49,target_h50,target_h51,target_h52,target_h53,target_h54,target_h55,target_h56,target_h57,target_h58,target_h59,target_h60,target_h61,target_h62,target_h63,target_h64,target_h65,target_h66,target_h67,target_h68,target_h69,target_h70,target_h71,target_h72,target_h73,target_h74,target_h75,target_h76,target_h77,target_h78,target_h79,target_h80,target_h81,target_h82,target_h83,target_h84,target_h85,target_h86,target_h87,target_h88,target_h89,target_h90,target_h91,target_h92,target_h93,target_h94,target_h95,target_h96
feature,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
nsw_price_asinh_lag_1,6,4,4,4,4,2,2,4,4,4,4,2,3,4,4,4,6,7,10,8,7,13,12,11,12,12,12,15,12,16,19,24,24,24,26,27,23,22,27,26,30,35,39,32,30,42,42,41,40,37,39,38,38,37,48,40,41,45,49,52,46,54,52,54,57,50,58,58,55,55,60,83,66,67,62,80,73,72,74,76,74,80,91,110,81,83,92,95,95,110,102,108,89,115,112,91
nsw_price_asinh_lag_2,9,6,6,6,6,6,6,6,6,6,5,6,6,7,7,7,10,11,12,13,14,15,11,14,14,15,19,14,20,15,24,27,25,27,25,25,24,33,25,34,34,36,40,41,34,41,37,39,42,44,42,47,45,46,45,48,50,53,55,56,53,56,46,51,56,54,50,50,52,54,55,63,67,61,56,69,79,77,73,64,80,75,81,106,82,91,79,94,109,89,77,93,109,111,98,104
nsw_price_asinh_lag_4,13,12,12,12,12,12,12,11,9,11,12,13,12,12,15,15,18,18,18,19,19,18,19,21,20,22,25,24,24,25,29,29,28,29,33,35,36,38,40,40,37,40,45,44,42,43,47,48,48,49,50,53,56,55,53,50,52,62,64,59,57,57,57,62,62,55,60,66,61,63,72,58,85,79,97,67,95,85,83,78,79,89,99,109,97,88,85,104,118,102,120,113,119,110,118,111
nsw_price_asinh_lag_12,26,23,22,21,19,23,22,24,25,28,25,27,25,25,25,27,29,29,31,35,36,36,39,35,37,35,37,41,43,44,48,45,47,50,50,54,51,53,57,58,57,60,58,58,58,60,58,62,61,60,59,63,62,62,63,64,66,67,66,66,67,66,69,67,71,91,74,74,75,91,90,98,103,93,96,105,110,106,124,118,112,112,110,111,110,98,117,117,121,112,100,87,112,113,116,115
nsw_price_asinh_lag_48,159,154,152,148,144,150,150,146,134,141,137,142,140,130,120,127,120,126,122,130,123,122,120,133,131,125,118,125,115,137,144,128,118,131,140,131,149,134,131,129,134,135,131,132,132,132,130,125,132,121,109,116,132,128,121,123,120,102,119,104,112,118,115,101,110,102,114,105,94,92,107,94,109,113,111,99,97,114,94,104,81,110,76,86,78,95,93,91,99,78,85,80,82,84,76,80
nsw_price_asinh_lag_96,235,223,224,214,212,212,211,208,197,208,206,191,199,193,196,197,186,183,173,154,162,156,151,139,156,143,149,149,122,131,116,116,113,112,114,110,104,105,106,103,99,99,91,77,88,94,85,85,75,73,79,82,80,78,85,70,81,81,80,84,80,82,93,69,77,76,81,91,95,89,85,89,97,105,92,114,111,95,106,112,117,126,126,116,113,127,118,123,107,123,123,135,125,136,117,122
nsw_price_asinh_lag_336,211,198,198,191,195,192,199,180,194,205,192,186,188,192,177,186,199,202,199,184,196,185,194,188,193,194,183,182,190,177,192,187,174,180,183,183,182,182,173,176,187,192,177,176,160,168,174,169,170,162,166,154,152,141,148,156,163,140,159,129,140,146,131,132,126,127,129,145,136,132,137,129,136,132,139,126,128,130,133,126,134,135,131,126,128,122,126,137,125,129,130,125,134,126,134,120
nsw_price_asinh_lag_335,202,199,186,195,193,196,202,204,195,188,187,199,190,180,188,198,200,197,188,193,199,198,187,195,200,192,179,190,189,199,174,192,160,170,189,188,198,177,199,181,194,188,196,201,170,181,178,148,184,169,159,171,164,165,160,155,155,155,161,155,144,156,150,150,129,151,152,135,151,137,138,133,134,126,137,130,135,133,137,139,130,124,124,133,133,131,133,131,134,131,138,134,135,117,139,128
nsw_price_asinh_lag_337,205,1

,feature,target_h1,target_h2,target_h3,target_h4,target_h5,target_h6,target_h7,target_h8,target_h9,target_h10,target_h11,target_h12,target_h13,target_h14,target_h15,target_h16,target_h17,target_h18,target_h19,target_h20,target_h21,target_h22,target_h23,target_h24,target_h25,target_h26,target_h27,target_h28,target_h29,target_h30,target_h31,target_h32,target_h33,target_h34,target_h35,target_h36,target_h37,target_h38,target_h39,target_h40,target_h41,target_h42,target_h43,target_h44,target_h45,target_h46,target_h47,target_h48,target_h49,target_h50,target_h51,target_h52,target_h53,target_h54,target_h55,target_h56,target_h57,target_h58,target_h59,target_h60,target_h61,target_h62,target_h63,target_h64,target_h65,target_h66,target_h67,target_h68,target_h69,target_h70,target_h71,target_h72,target_h73,target_h74,target_h75,target_h76,target_h77,target_h78,target_h79,target_h80,target_h81,target_h82,target_h83,target_h84,target_h85,target_h86,target_h87,target_h88,target_h89,target_h90,target_h91,target_h92,target_h93,target_h94,target_h95,target_h96
0,qld_price_asinh_lag_2,29,26,27,26,30,30,35,34,33,34,32,36,37,39,40,41,44,45,47,44,48,50,53,58,56,59,66,64,61,60,66,69,75,71,73,71,67,73,72,72,78,78,84,89,90,92,102,90,101,93,95,114,122,108,129,136,128,136,129,132,130,133,138,142,151,154,155,149,161,170,172,167,166,160,182,155,170,157,157,153,166,193,176,201,202,175,166,180,178,190,171,171,186,169,162,164
1,qld_price_asinh_lag_335,265,248,262,259,243,234,250,257,244,257,275,248,257,263,238,278,264,306,280,299,263,274,279,265,294,286,287,284,272,275,277,273,239,272,270,282,268,260,276,265,279,254,259,270,231,229,245,276,233,236,233,218,250,252,246,231,233,257,264,235,240,265,259,217,242,255,245,206,223,229,227,231,214,186,212,244,218,222,219,189,226,189,215,216,186,214,205,200,220,183,225,196,206,219,228,215
2,qld_price_asinh_lag_337,276,264,274,272,271,275,266,283,270,276,264,285,268,283,252,293,273,280,274,292,299,294,280,300,287,307,320,318,281,292,307,279,324,286,292,296,310,275,290,296,276,279,253,260,277,284,295,302,300,254,265,293,263,263,252,274,284,261,273,252,245,236,264,249,215,247,243,237,236,221,218,212,229,204,229,246,261,208,237,207,212,259,213,199,215,249,213,186,211,213,212,214,213,213,206,212
3,sa_price_asinh_lag_2,109,103,106,108,108,113,123,117,127,117,114,137,127,143,130,142,146,143,154,174,183,173,183,182,181,186,196,188,201,202,230,225,213,226,234,210,240,262,243,235,224,261,237,234,255,255,256,271,288,253,285,313,288,271,293,266,289,301,304,294,298,291,283,266,291,282,300,281,286,293,318,294,303,308,302,304,336,297,292,304,292,285,297,292,294,325,310,322,333,305,325,292,319,320,300,311
4,sa_price_asinh_lag_337,412,400,403,391,395,401,416,408,402,403,390,406,407,404,394,390,403,408,394,399,393,388,392,395,403,386,390,390,389,408,385,389,400,389,394,399,398,405,408,390,397,397,397,405,397,392,394,390,386,390,377,412,391,391,394,395,380,380,398,388,393,389,388,385,384,384,379,378,384,375,369,386,367,373,368,382,371,380,377,359,376,367,355,368,381,361,377,368,373,381,366,374,383,380,370,370
5,vic_price_asinh_lag_2,60,62,59,64,66,66,72,74,76,73,68,73,90,90,97,92,97,100,103,107,109,104,105,108,109,112,125,115,136,121,126,138,141,147,153,151,153,147,157,162,149,164,153,174,182,170,185,179,185,165,197,191,182,179,180,172,180,183,196,181,193,190,187,183,213,204,244,212,199,252,278,242,224,242,206,219,227,266,207,233,214,224,250,241,242,211,222,256,250,244,241,240,234,208,216,207
6,vic_price_asinh_lag_335,336,348,346,341,336,340,363,364,358,360,336,350,346,349,347,333,346,355,330,339,330,337,337,338,333,345,331,334,337,348,349,354,345,344,338,365,348,335,327,331,328,336,332,353,349,341,342,347,341,320,324,348,341,351,345,338,330,360,344,332,351,342,322,329,338,328,322,324,300,353,350,329,347,304,318,320,294,326,296,312,316,298,298,316,327,322,308,315,308,323,291,286,308,343,353,337
7,vic_price_asinh_lag_337,346,319,339,358,318,341,354,353,350,347,352,357,350,336,329,356,356,368,328,359,346,373,332,359,342,339,344,357,353,363,35

171


Export

In [13]:
features_ranked_ordered_duplicate.to_csv("features_ranked_ordered_duplicate.csv")

In [14]:
features_ranked_ordered_unique.to_parquet(variables.FEATURES_RANKED_ORDERED_UNIQUE_PATH)
features_unique_data.to_parquet(variables.FEATURES_UNIQUE_DATA_PATH)